# Delivery-residual validation & verdict-specific repairs

Notebook wrapper for the two experiment suites (`obstruction.py`, `repairs.py`).

- **Experiment 1** — constrained delivery residual `R(S)`: class separation, verdict agreement, prediction of ablation effects.
- **Experiment 2** — repair matrix: verdict-specific interventions with cross-class controls.

Requirements: `torch`, `transformers`, `scipy`, `pandas`, `matplotlib`. Run on GPU for real models (fp32 + eager attention). The tiny-model smoke test at the bottom runs on CPU in seconds.

**Cohort**: point `COHORT_PATH` at your exported verdict-tier jsonl (schema in `README.md`). If it is `None`, a small demo cohort is built from capital-city prompts and labeled with a *rough stand-in* for the paper's verdict rule — replace with your real labels for any result you intend to report.

In [ ]:
# ---------------- configuration ----------------
MODEL_NAME  = "meta-llama/Llama-3.2-3B"   # any Llama/Qwen2.5-style decoder
COHORT_PATH = None                         # e.g. "cohorts/llama32_3b_parametric.jsonl"
OUT_DIR     = "results/notebook_run"
DEVICE      = "cuda"                       # "cpu" works for small models

# science knobs (see README: decide & report)
PIN_FRAC    = 0.5      # source section pinned through PIN_FRAC * L layers
LAM         = 30.0     # terminal delivery row weight
C_PLUS_Q    = 0.10     # success quantile for c_plus (paper's Q+_{0.10};
                       #  0.5 = median, stricter but mislabels weak successes)
K_EDGES     = 8        # transport repair: top-k tDLA edges
K_HEADS     = 4        # selection repair: top-k demoter heads
RUN_ABLATION_SCREEN = True   # fast frozen-account ablation effects (exp 1c)

import os, json, torch
os.makedirs(OUT_DIR, exist_ok=True)
torch.set_grad_enabled(False);

In [ ]:
# ---------------- model ----------------
from transformers import AutoModelForCausalLM, AutoTokenizer
from frozen_cache import Weights

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32,
    attn_implementation="eager").to(DEVICE).eval()   # eager is REQUIRED (C1)
W = Weights(model)
print(f"{MODEL_NAME}: L={W.L} H={W.H} d_head={W.dh}")

## Cohort

Each record: `prompt`, `target_first_token`, `competitor_token`, `source_span` (inclusive, contiguous, model tokenizer), `verdict`, optional `tau`, `ablation_effect`, `donor_prompt`/`donor_source_span`.

In [ ]:
def find_span(prompt, substring, tokenizer):
    """Inclusive token-index span of `substring` in `prompt` (fast tokenizer)."""
    enc = tokenizer(prompt, return_offsets_mapping=True)
    a = prompt.index(substring); b = a + len(substring)
    idx = [i for i, (s, e) in enumerate(enc.offset_mapping) if s < b and e > a]
    return [min(idx), max(idx)]

if COHORT_PATH is not None:
    records = [json.loads(l) for l in open(COHORT_PATH)]
    print(f"loaded {len(records)} records")
else:
    # -------- demo cohort (replace with your verdict-tier export) --------
    DEMO = [("The capital of Maryland is", "Maryland", " Annapolis"),
            ("The capital of France is", "France", " Paris"),
            ("The capital of Australia is", "Australia", " Canberra"),
            ("The capital of Canada is", "Canada", " Ottawa"),
            ("The capital of Burkina Faso is", "Burkina Faso", " Ouagadougou"),
            ("The capital of Kiribati is", "Kiribati", " Tarawa"),
            ("The capital of Eswatini is", "Eswatini", " Mbabane"),
            ("The capital of Vanuatu is", "Vanuatu", " Port Vila")]
    records = []
    for prompt, subject, answer in DEMO:
        g = tok(answer, add_special_tokens=False).input_ids[0]
        ids = tok(prompt, return_tensors="pt").input_ids.to(DEVICE)
        top = int(model(ids).logits[0, -1].argmax())
        records.append(dict(prompt=prompt, target_first_token=g,
                            competitor_token=top if top != g else -1,
                            source_span=find_span(prompt, subject, tok),
                            verdict="correct" if top == g else None,
                            tau=None, ablation_effect=None))
    print(f"demo cohort: {len(records)} records "
          f"({sum(r['verdict']=='correct' for r in records)} correct)")

In [ ]:
# ---------------- pass 1: caches, certificates, c_plus ----------------
from frozen_cache import build_cache, certify_frozen, target_delivery

caches, delivered = {}, []
for i, r in enumerate(records):
    ids = tok(r["prompt"], return_tensors="pt").input_ids[0].to(DEVICE)
    C = build_cache(model, W, ids)      # C1 inside
    certify_frozen(W, C)                # C2
    caches[i] = (ids, C)
    if r["verdict"] == "correct":
        delivered.append(float(target_delivery(C.resid[-1], W, C,
                                               r["target_first_token"])))
c_plus = float(torch.tensor(delivered).quantile(C_PLUS_Q))
print(f"C1/C2 pass on all {len(records)} prompts; "
      f"c_plus = {c_plus:.3f} (Q_{C_PLUS_Q:.2f} over {len(delivered)} successes)")

In [ ]:
# ---------------- stand-in verdicts (DEMO ONLY) ----------------
# Rough version of the paper's ordered rule, used only when the cohort has no
# labels. Your exported verdict-tier labels are the ground truth.
# Ordered rule (mirrors the paper): delivered at success level -> selection;
# else source content at success level -> transport; else source.
from frozen_cache import tdla_edge_scores, target_delivery

if any(r["verdict"] is None for r in records):
    ug = lambda C, t: ((C.inv_f[-1] * W.ln_f) * W.WU[t]).squeeze()
    def pi_S(C, t, S):
        u = ug(C, t)
        lo, hi = W.L // 4, 3 * W.L // 4
        return max(float((C.resid[l][j] * u).sum())
                   for l in range(lo, hi + 1) for j in S)
    succ = [i for i, r in enumerate(records) if r["verdict"] == "correct"]
    q10 = lambda vals: float(torch.tensor(vals).quantile(0.10))
    span = lambda r: range(r["source_span"][0], r["source_span"][1] + 1)
    pis  = [pi_S(caches[i][1], records[i]["target_first_token"],
                 span(records[i])) for i in succ]
    dels = [float(target_delivery(caches[i][1].resid[-1], W, caches[i][1],
                  records[i]["target_first_token"])) for i in succ]
    th_pi, th_del = q10(pis), q10(dels)
    for i, r in enumerate(records):
        if r["verdict"] is not None: continue
        S = span(r)
        p = pi_S(caches[i][1], r["target_first_token"], S)
        d = float(target_delivery(caches[i][1].resid[-1], W, caches[i][1],
                                  r["target_first_token"]))
        r["tau"] = float(tdla_edge_scores(W, caches[i][1],
                         r["target_first_token"], S).sum())
        r["verdict"] = ("selection" if d >= th_del else
                        "transport" if p >= th_pi else "source")
    from collections import Counter
    print("stand-in verdicts:", Counter(r["verdict"] for r in records))

## Experiment 1 — the pinned obstruction `R(S)`

In [ ]:
from obstruction import solve_obstruction
from run_obstruction_validation import ablation_effect
import pandas as pd

rows, profiles = [], {}
for i, r in enumerate(records):
    ids, C = caches[i]
    S = list(range(r["source_span"][0], r["source_span"][1] + 1))
    res = solve_obstruction(W, C, S, r["target_first_token"], c_plus,
                            pin_L=int(PIN_FRAC * W.L), lam=LAM)
    eff = r.get("ablation_effect")
    if eff is None and RUN_ABLATION_SCREEN and r["verdict"] != "correct":
        eff = ablation_effect(model, ids, W, C, r["target_first_token"], S)
    attn_mass = float(sum(C.layers[l].A[:, -1, S].sum() for l in range(W.L)))
    rows.append(dict(idx=i, verdict=r["verdict"], ob=res.ob, ob_norm=res.ob_norm,
                     centroid=res.depth_centroid, gap0=res.delivered_gap0,
                     pi_S=res.pi_S, tau=r.get("tau"), attn_mass=attn_mass,
                     ablation_effect=eff, cg_iters=res.cg_iters))
    profiles[i] = res.profile
    print(f"[{i:3d}] {r['verdict']:<9s} ob={res.ob:8.3f} centroid={res.depth_centroid:.2f}")

df = pd.DataFrame(rows)
df.to_csv(f"{OUT_DIR}/obstruction.csv", index=False)
df.groupby("verdict")[["ob", "ob_norm", "centroid"]].median()

In [ ]:
# ---- 1a/1b: separation + verdict agreement ----
from run_obstruction_validation import auc

g = lambda v, k="ob": df[df.verdict == v][k].tolist()
print(f"AUC transport vs correct+selection      : "
      f"{auc(g('transport'), g('correct') + g('selection')):.3f}")
print(f"AUC {{source,transport}} vs {{sel,corr}} : "
      f"{auc(g('source') + g('transport'), g('selection') + g('correct')):.3f}")
print(f"AUC source vs transport (centroid)    : "
      f"{auc(g('transport', 'centroid'), g('source', 'centroid')):.3f}")

fails = df[df.verdict != "correct"]
if len(fails):
    med = fails.ob.median()
    pred = (fails.ob > med) & (fails.centroid > 0.35)
    print(f"ob-threshold agreement with transport verdicts: "
          f"{(pred == (fails.verdict == 'transport')).mean():.2f}")

In [ ]:
# ---- plots ----
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
order = [v for v in ("source", "transport", "selection", "correct")
         if v in set(df.verdict)]
ax[0].boxplot([g(v) for v in order], labels=order)
ax[0].set_ylabel("R(S)"); ax[0].set_title("obstruction by verdict")
for i, r in df.iterrows():
    p = profiles[r.idx]
    ax[1].plot(torch.arange(1, W.L + 1) / W.L, p / (p.sum() + 1e-30),
               alpha=0.6, label=r.verdict)
ax[1].set_xlabel("relative depth"); ax[1].set_ylabel("residual energy share")
ax[1].set_title("obstruction depth profiles")
h, l = ax[1].get_legend_handles_labels()
uniq = dict(zip(l, h)); ax[1].legend(uniq.values(), uniq.keys(), fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/obstruction.png", dpi=150); plt.show()

In [ ]:
# ---- 1c: predicting ablation effect sizes ----
from scipy.stats import spearmanr

have = df.dropna(subset=["ablation_effect"])
if len(have) >= 5:
    for name in ("ob", "attn_mass", "tau"):
        sub = have.dropna(subset=[name])
        if len(sub) >= 5:
            rho, p = spearmanr(sub[name], sub.ablation_effect)
            print(f"Spearman({name:>9s}, effect) = {rho:+.3f}  (p={p:.1e}, n={len(sub)})")
else:
    print("too few ablation effects; pass your Sec 6.9 values via the cohort")

## Experiment 2 — the repair matrix

Diagonal should dominate; `random_heads` column should sit near zero.

In [ ]:
from repairs import (repair_source, repair_transport, repair_selection,
                     repair_random_heads, certify_wrappers, margin)

certify_wrappers(model, W, caches[0][0])   # C-live
BAND = list(range(int(0.2 * W.L), int(0.6 * W.L)))

rrows = []
for i, r in enumerate(records):
    if r["verdict"] == "correct" or r["competitor_token"] == -1:
        continue
    ids, C = caches[i]
    gtok, ctok = r["target_first_token"], r["competitor_token"]
    S = list(range(r["source_span"][0], r["source_span"][1] + 1))
    m0, _ = margin(model, ids, gtok, ctok)
    out = dict(idx=i, verdict=r["verdict"], margin_base=m0)
    if r.get("donor_prompt"):
        d_ids = tok(r["donor_prompt"], return_tensors="pt").input_ids[0].to(DEVICE)
        D = build_cache(model, W, d_ids)
        dS = list(range(r["donor_source_span"][0], r["donor_source_span"][1] + 1))
        donor = {l: D.resid[l][dS] for l in BAND}
        m, f = repair_source(model, ids, gtok, ctok, S, donor, BAND)
        out["source_patch"], out["source_patch_flip"] = m - m0, f
    m, f = repair_transport(model, W, C, ids, gtok, ctok, S, k=K_EDGES)
    out["transport_edges"], out["transport_edges_flip"] = m - m0, f
    m, f = repair_selection(model, W, C, ids, gtok, ctok, k=K_HEADS)
    out["selection_demoters"], out["selection_demoters_flip"] = m - m0, f
    m, f = repair_random_heads(model, W, ids, gtok, ctok, k=K_HEADS, seed=i)
    out["random_heads"], out["random_heads_flip"] = m - m0, f
    rrows.append(out)

rdf = pd.DataFrame(rrows)
rdf.to_csv(f"{OUT_DIR}/repair_matrix.csv", index=False)
cols = [c for c in ("source_patch", "transport_edges",
                    "selection_demoters", "random_heads") if c in rdf]
matrix = rdf.groupby("verdict")[cols].mean()
flips = rdf.groupby("verdict")[[c + "_flip" for c in cols]].mean()
print("mean margin change:"); display(matrix.round(2))
print("flip rates:"); display(flips.round(2))

In [ ]:
# ---- heatmap ----
import numpy as np
fig, ax = plt.subplots(figsize=(6, 2.5))
im = ax.imshow(matrix.values, cmap="RdBu_r",
               vmin=-np.abs(matrix.values).max(), vmax=np.abs(matrix.values).max())
ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=20)
ax.set_yticks(range(len(matrix.index)), matrix.index)
for y in range(matrix.shape[0]):
    for x in range(matrix.shape[1]):
        ax.text(x, y, f"{matrix.values[y, x]:+.2f}", ha="center", va="center")
plt.colorbar(im, label="mean margin change")
ax.set_title("repair matrix (diagonal should dominate)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/repair_matrix.png", dpi=150); plt.show()

## Cross-model aggregation

Rerun the notebook per model with a distinct `OUT_DIR`, then concatenate:

```python
import glob, pandas as pd
frames = [pd.read_csv(p).assign(model=p.split("/")[-2])
          for p in glob.glob("results/*/obstruction.csv")]
pd.concat(frames).groupby(["model", "verdict"]).ob.median().unstack()
```

## Smoke test (CPU, seconds)

Verifies all certificates on a tiny random model — run this first on any new
environment or transformers version.

In [ ]:
# Runs in the SAME python env as this kernel (avoids the
# ModuleNotFoundError from a system python without torch/transformers):
import sys, subprocess
print(subprocess.run([sys.executable, "smoke_test.py"],
                     capture_output=True, text=True).stdout[-2000:])